# EDA & Representation Experiments — ESOL Solubility Dataset

This notebook is for **exploration**, not production logic (see `ml/` for the actual
pipeline used by `scripts/train_model.py` and the backend). It:

1. Loads and inspects the raw ESOL dataset
2. Looks at the distribution of the target and how it relates to a few descriptors
3. Compares the scaffold split vs. the random split
4. Compares the descriptor-based representation against Morgan fingerprints with a
   quick cross-validated model, to justify the representation choice documented in
   `ml/features/fingerprints.py`

> **Note:** this notebook was authored in a sandboxed environment without RDKit
> installed, so it has **not been executed** and carries no saved outputs. Run it
> top-to-bottom locally (`pip install -r requirements.txt && jupyter lab`) to populate
> real plots and numbers — everything below is real, runnable code, just not yet run.


In [ ]:
import sys
from pathlib import Path

# Allow `import ml...` when running this notebook from notebooks/
sys.path.insert(0, str(Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from ml import config
from ml.data.loader import load_and_clean
from ml.data.validation import validate_dataframe
from ml.data.splitting import assert_no_leakage, random_split, scaffold_split
from ml.features.descriptors import compute_descriptors_for_dataframe
from ml.features.fingerprints import compute_fingerprints_for_dataframe
from ml.pipeline import build_feature_matrix

sns.set_theme(style="whitegrid")
RANDOM_SEED = config.RANDOM_SEED


## 1. Load and clean

In [ ]:
df = load_and_clean()
print(f"{len(df)} rows after cleaning")
df.head()


In [ ]:
valid_df, invalid_df = validate_dataframe(df)
print(f"{len(valid_df)} valid molecules, {len(invalid_df)} invalid (dropped)")
if len(invalid_df):
    display(invalid_df)


## 2. Target distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.histplot(valid_df[config.TARGET_COL], bins=40, kde=True, ax=ax)
ax.set_xlabel("measured log(solubility) [mol/L]")
ax.set_title(f"Target distribution (n={len(valid_df)})")
plt.show()

valid_df[config.TARGET_COL].describe()


## 3. Descriptors vs. target

Quick sanity check that the chosen descriptor set actually correlates with the
target the way the domain knowledge in `ml/features/descriptors.py` claims it should
(most importantly: MolLogP should correlate *negatively* and *strongly* with
solubility — this is the best-known single predictor in the literature).

In [ ]:
X, y = build_feature_matrix(valid_df)
feat_df = X.copy()
feat_df[config.TARGET_COL] = y.values

corr = feat_df.corr(numeric_only=True)[config.TARGET_COL].drop(config.TARGET_COL).sort_values()
fig, ax = plt.subplots(figsize=(6, 5))
corr.plot(kind="barh", ax=ax)
ax.set_xlabel(f"Correlation with {config.TARGET_COL}")
ax.set_title("Descriptor correlation with measured solubility")
plt.tight_layout()
plt.show()

corr


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(feat_df["MolLogP"], feat_df[config.TARGET_COL], alpha=0.4, s=15)
ax.set_xlabel("MolLogP")
ax.set_ylabel("measured log(solubility) [mol/L]")
ax.set_title("MolLogP vs. measured solubility (expect a strong negative trend)")
plt.show()


## 4. Scaffold split vs. random split

Sanity-checks the claim made in `ml/data/splitting.py`'s docstring: a random split
should look "easier" (lower held-out error) than a scaffold split on the same model,
because random splitting lets near-duplicate analogues leak across train/test.

In [ ]:
from sklearn.linear_model import Ridge
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

def quick_ridge_score(train_df, test_df):
    X_train, y_train = build_feature_matrix(train_df)
    X_test, y_test = build_feature_matrix(test_df)
    model = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0, random_state=RANDOM_SEED))])
    model.fit(X_train, y_train)
    return mean_absolute_error(y_test, model.predict(X_test))

scaffold_train, scaffold_val, scaffold_test = scaffold_split(valid_df, seed=RANDOM_SEED)
random_train, random_val, random_test = random_split(valid_df, seed=RANDOM_SEED)

assert_no_leakage(scaffold_train, scaffold_val, scaffold_test)
assert_no_leakage(random_train, random_val, random_test)

scaffold_mae = quick_ridge_score(scaffold_train, scaffold_test)
random_mae = quick_ridge_score(random_train, random_test)

print(f"Ridge test MAE — scaffold split: {scaffold_mae:.3f}")
print(f"Ridge test MAE — random split:   {random_mae:.3f}")
print("(Expect scaffold MAE >= random MAE: scaffold splitting is the harder, more honest evaluation.)")


## 5. Representation comparison: descriptors vs. Morgan fingerprints

A quick, informal comparison (5-fold CV on the training split, one fixed Ridge
config — not a full hyperparameter search) to check whether fingerprints are worth
the interpretability trade-off documented in `ml/features/fingerprints.py`. The
model actually shipped by `scripts/train_model.py` uses the descriptor
representation unless this comparison shows a material accuracy gain from
fingerprints.

In [ ]:
from sklearn.model_selection import cross_val_score, KFold

X_desc, y_train_full = build_feature_matrix(scaffold_train)
fp_df = compute_fingerprints_for_dataframe(scaffold_train)

cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
ridge = Pipeline([("scaler", StandardScaler()), ("ridge", Ridge(alpha=1.0, random_state=RANDOM_SEED))])

desc_scores = -cross_val_score(ridge, X_desc, y_train_full, cv=cv, scoring="neg_mean_absolute_error")
fp_scores = -cross_val_score(ridge, fp_df, y_train_full, cv=cv, scoring="neg_mean_absolute_error")

print(f"Descriptors — CV MAE: {desc_scores.mean():.3f} +/- {desc_scores.std():.3f}")
print(f"Fingerprints — CV MAE: {fp_scores.mean():.3f} +/- {fp_scores.std():.3f}")


**Decision rule applied in `scripts/train_model.py`:** ship the descriptor-based
model unless fingerprints beat it by a margin clearly larger than the CV
standard deviation above — a small, noise-level improvement isn't worth losing
interpretability (the descriptor model can say *why* a prediction changed; a
1024-bit fingerprint model cannot).

## 6. A few example structures

In [ ]:
from rdkit import Chem
from rdkit.Chem import Draw

example_smiles = valid_df[config.CANONICAL_SMILES_COL].sample(6, random_state=RANDOM_SEED).tolist()
mols = [Chem.MolFromSmiles(s) for s in example_smiles]
Draw.MolsToGridImage(mols, molsPerRow=3, subImgSize=(220, 180), legends=example_smiles)
